Kaarina Mattika, CS87A, Section 1758 https://github.com/kmattika/cs82a-portfolio

In [1]:
import pandas as pd
import seaborn as sns

In [2]:
df = pd.read_csv(r'C:\Users\kamat\OneDrive\Desktop\CS82A\usarrests.csv')

In [3]:
display(df.head(10))
display(df.shape)
display(df.info())
display(df.describe())

,Unnamed: 0,Murder,Assault,UrbanPop
0,Alabama,13.2,236.0,58
1,Alaska,10.0,263.0,48
2,Arizona,8.1,294.0,80
3,Arkansas,8.8,190.0,50
4,California,9.0,276.0,91
5,Colorado,7.9,204.0,78
6,Connecticut,3.3,110.0,77
7,Delaware,5.9,238.0,72
8,Florida,15.4,335.0,80
9,Georgia,17.4,NaN,60


(50, 4)

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  50 non-null     str    
 1   Murder      50 non-null     float64
 2   Assault     49 non-null     float64
 3   UrbanPop    50 non-null     int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 2.1 KB


None

,Murder,Assault,UrbanPop
count,50.00000,49.000000,50.00000
mean,7.78800,182.183673,74.20000
std,4.35551,130.877435,73.40828
min,0.80000,45.000000,6.00000
25%,4.07500,109.000000,53.25000
50%,7.25000,159.000000,66.00000
75%,11.25000,249.000000,77.75000
max,17.40000,879.000000,570.00000


Immediately, we can see that Georgia has a missing value under 'Assault'. We can also discern that Georgia's values under 'Murder' and 'UrbanPop' are within expected bounds; the value under 'Murder' is 2.21 standard deviations from the mean, and the value under 'UrbanPop' is within one standard deviation. My resolution is to fill Georgia's 'Assault' value with the median value of that column once cleaned.

In [4]:
df[df['UrbanPop'] > 100]

,Unnamed: 0,Murder,Assault,UrbanPop
14,Iowa,2.2,56.0,570


Above, we see that Iowa's UrbanPop value is '570'. This is an impossible value. A proportion value cannot exceed 100, as you cannot have more than 100% of a value occuring in a fixed population. The error here is most likely a clerical input, with a zero accidentally appended following the correct figure of 57. The Iowa State Data Center's current projections show that the 2020 US Census data lists Iowa's urban population at 63.2% (https://www.iowadatacenter.org/index.php/data-by-source/decennial-census/urban-and-rural-population); the six percentage points between the data stated figure of 57 and the 2020 Census' figure of 63 suggests that a difference in methodology between state agencies could be the reason between these differences. My resolution is to amend row 14's figure of 570 in 'UrbanPop' to 57.

In [5]:
df[df['UrbanPop'] < 10]

,Unnamed: 0,Murder,Assault,UrbanPop
31,New York,11.1,254.0,6


We have a slightly different issue with the above data. We see that the 'UrbanPop' value associated with New York is listed as '6', the lowest score in this column. Heuristically, we know this is not true - New York City alone has a greater population that some states in this dataset, and given that this dataset is comprised of state data, we know that this cannot be true. As a result, my resolution is to remote row 31, as we cannot trust this row's data.

In [6]:
df[df['Assault'] > 300]

,Unnamed: 0,Murder,Assault,UrbanPop
8,Florida,15.4,335.0,80
32,North Carolina,13.0,337.0,45
39,South Carolina,14.4,879.0,48


Here, we see South Carolina's Assault value as 879. This is an improbable value. It is more than double the two closest states, Florida and North Carolina, despite having similar values in 'Murder' and proportionately accurate values in 'UrbanPop'. The value of 879 is also more than five standard deviations from the mean value of 182, making the statistical expectation of 879 less than 0.0001%. I do not have any plausible theories as to the source of this faulty data, but from a statistical standpoint, it is extremely improbable that this value is genuine. My resolution is to remove row 39, as we cannot trust the source of this row's data.

In [7]:
df[df['Murder'] < 2.5]

,Unnamed: 0,Murder,Assault,UrbanPop
14,Iowa,2.2,56.0,570
18,Maine,2.1,83.0,51
28,New Hampshire,2.1,57.0,56
33,North Dakota,0.8,45.0,44
44,Vermont,2.2,48.0,32


Now, we have North Dakota's value for murder at '0.8', which is satistically within the expected range of values if given a mean of 7.788 and a standard deviation of 4.355. However, the value appears to be abnormal given that it is the only value under 2.0 in this column, and appears disproportional to other corresponding values in other rows. While I don't know how to create a multiple linear regression model in Jupyter yet, from a visual examination of the data, we can see that 0.8 appears disproportional to it's complimentary values.

In [8]:
display(df.iloc[[22, 25, 33, 40, 49]])

,Unnamed: 0,Murder,Assault,UrbanPop
22,Minnesota,2.7,72.0,66
25,Montana,6.0,109.0,53
33,North Dakota,0.8,45.0,44
40,South Dakota,3.8,86.0,45
49,Wyoming,6.8,161.0,60


Here, I've pulled the states geographically closest to North Dakota. None of them have values in 'Murder' remotely close.

In [9]:
df[(df["Assault"] >= 0) & (df["Assault"] <= 110)]

,Unnamed: 0,Murder,Assault,UrbanPop
6,Connecticut,3.3,110.0,77
10,Hawaii,5.3,46.0,83
14,Iowa,2.2,56.0,570
16,Kentucky,9.7,109.0,52
18,Maine,2.1,83.0,51
22,Minnesota,2.7,72.0,66
25,Montana,6.0,109.0,53
26,Nebraska,4.3,102.0,62
28,New Hampshire,2.1,57.0,56
33,North Dakota,0.8,45.0,44


Here, I've pulled the values under 'Assault' to 0.5 standard deviations away from North Dakota's value under 'Assault'. Of all closely comparable values (Hawaii, Vermont, and Wisconsin), each have values over 2.0 under 'Murder'). My best guess is that this is a decimal placement error, and the true value is 80, not 0.8, which I will apply during my data cleaning.

Data cleaning notes, in order:

Amended row 14's figure of 570 in 'UrbanPop' to 57, to correct an assumed typo.
Removed row 31, as we cannot trust the data source.
Removed row 39, as we cannot trust the data source.
Amended row 33's figure of 0.8 in 'Murder' to 8.0, to correct an assumed typo.
Filled row 9's 'Assault' value with the column's median value.

In [10]:
df.loc[14, 'UrbanPop'] = 57
df = df.drop([31, 39])
df.loc[33, 'Murder'] = 8.0
fill_value = df['Assault'].median()
df['Assault'] = df['Assault'].fillna(fill_value)

In [11]:
display(df.head(10))
display(df.shape)
display(df.info())
display(df.describe())

,Unnamed: 0,Murder,Assault,UrbanPop
0,Alabama,13.2,236.0,58
1,Alaska,10.0,263.0,48
2,Arizona,8.1,294.0,80
3,Arkansas,8.8,190.0,50
4,California,9.0,276.0,91
5,Colorado,7.9,204.0,78
6,Connecticut,3.3,110.0,77
7,Delaware,5.9,238.0,72
8,Florida,15.4,335.0,80
9,Georgia,17.4,156.0,60


(48, 4)

<class 'pandas.DataFrame'>
Index: 48 entries, 0 to 49
Data columns (total 4 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   Unnamed: 0  48 non-null     str    
 1   Murder      48 non-null     float64
 2   Assault     48 non-null     float64
 3   UrbanPop    48 non-null     int64  
dtypes: float64(2), int64(1), str(1)
memory usage: 3.3 KB


None

,Murder,Assault,UrbanPop
count,48.000000,48.000000,48.000000
mean,7.731250,165.625000,65.479167
std,4.191996,82.384833,14.247045
min,2.100000,45.000000,32.000000
25%,4.225000,108.250000,55.500000
50%,7.250000,156.000000,66.000000
75%,10.625000,240.750000,77.250000
max,17.400000,337.000000,91.000000


After our cleaning, our data looks much more reasonable. Our expected values in 'Murder', 'Assault', and 'UrbanPop' all have maximum and minimum values within expected statistical variance. Using a combination of statistical analysis, heuristical knowledge, and publicly available reference information, we were able to successfully clean this data for use.